# Day 9 of ML 30-Days Challenge

# Task

**Feature Scaling**: Understand StandardScaler and MinMaxScaler. Learn why scaling is critical for distance-based algorithms (like SVM and K-NN) and how it helps gradient-based optimization algorithms converge faster.

## 1. Setup: Creating a Sample Dataset

In [1]:
import pandas as pd

# Sample data
data = {
    'Country': ['USA', 'India', 'Germany', 'USA', 'India'],
    'Education': ['High School', 'Graduate', 'Masters', 'Graduate', 'High School'],
    'Salary': [70000, 50000, 120000, 95000, 45000]
}

df = pd.DataFrame(data)
print(df)

   Country    Education  Salary
0      USA  High School   70000
1    India     Graduate   50000
2  Germany      Masters  120000
3      USA     Graduate   95000
4    India  High School   45000


## 2. Implementing Label Encoding (for Ordinal Data)

In [2]:
from sklearn.preprocessing import LabelEncoder

# Map Education manually in the order: High School < Graduate < Masters
education_order = {'High School': 0, 'Graduate': 1, 'Masters': 2}
df['Education_encoded'] = df['Education'].map(education_order)

print(df[['Education', 'Education_encoded']])

     Education  Education_encoded
0  High School                  0
1     Graduate                  1
2      Masters                  2
3     Graduate                  1
4  High School                  0


## 3. Implementing One-Hot Encoding (for Nominal Data)

### Simple Way (Pandas)

In [3]:
# Using pd.get_dummies
df_dummies = pd.get_dummies(df, columns=['Country'], drop_first=True)
print(df_dummies) # just for dummies 😝

     Education  Salary  Education_encoded  Country_India  Country_USA
0  High School   70000                  0          False         True
1     Graduate   50000                  1           True        False
2      Masters  120000                  2          False        False
3     Graduate   95000                  1          False         True
4  High School   45000                  0           True        False


### Scikit-learn Way (for Pipelines)

In [4]:
from sklearn.preprocessing import OneHotEncoder

# Create OneHotEncoder instance
ohe = OneHotEncoder(drop='first', sparse_output=False)

# Fit and transform 'Country'
country_encoded = ohe.fit_transform(df[['Country']])

# Create new DataFrame from encoded array
encoded_df = pd.DataFrame(country_encoded, columns=ohe.get_feature_names_out(['Country']))
print(encoded_df)

   Country_India  Country_USA
0            0.0          1.0
1            1.0          0.0
2            0.0          0.0
3            0.0          1.0
4            1.0          0.0


## 4. Handling Mixed Data Types with ColumnTransformer

In [5]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler

# Original DataFrame again
X = df[['Country', 'Education', 'Salary']]

# ColumnTransformer setup
preprocessor = ColumnTransformer(
    transformers=[
        ('country_ohe', OneHotEncoder(drop='first'), ['Country']),
        ('education_ord', 'passthrough', ['Education_encoded']),
        ('num_scaler', StandardScaler(), ['Salary'])
    ],
    remainder='drop'
)

X_transformed = preprocessor.fit_transform(df)
print(X_transformed)

[[ 0.          1.          0.         -0.21293203]
 [ 1.          0.          1.         -0.92270547]
 [ 0.          0.          2.          1.56150157]
 [ 0.          1.          1.          0.67428477]
 [ 1.          0.          0.         -1.10014883]]


## 5. Integrating ColumnTransformer into a Pipeline

In [6]:
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression

# Dummy target column (for classification)
df['Target'] = [1, 0, 1, 1, 0]

# Pipeline with preprocessor and model
pipeline = Pipeline(steps=[
    ('preprocess', preprocessor),
    ('classifier', LogisticRegression())
])

# Fit pipeline
pipeline.fit(df, df['Target'])

# Predict
predictions = pipeline.predict(df)
print("Predictions:", predictions)

Predictions: [1 0 1 1 0]
